> 🔓 **SOLUTIONS EDITION**: this is the instructor copy with worked solutions and executed outputs. Please attempt every problem in the student notebook first; struggling is where learning happens.

# 🏗️ Project 3: Loan Approval Classification
*Phase 4 project · classification · a leakage trap · costs · fairness · reason codes*

| ⏱️ Time | 🎚️ Level | 🧰 Prerequisites |
|---|---|---|
| ~270 min | 🔴 Project | Lessons 09–16 (especially 10, 16) |

---
## 🎬 The Hook
**From:** Ms. Wambui, Risk Director, Jengo Microfinance  
**Subject:** Faster, fairer loan pre-screening

> *"Our officers review ~3 000 applications a year and decisions vary a lot between officers. We want a pre-screening model that recommends approve / decline, so officers only review borderline cases. Approving a loan that should have been declined costs us roughly **five times** as much as declining a good applicant. Regulators will ask us to explain individual declines and to show that we don't discriminate by gender or marital status. Can you help?"*

A loan model touches people's lives. This project is where *accuracy* meets **responsibility**: leakage, costs, fairness and explanations are not optional extras.

---
## 🎯 Learning Objectives
By the end of this notebook you will be able to:

1. Detect **target leakage** by interrogating a suspicious column, and quantify how misleading it is.
2. Build a leakage-free, fair model tournament for a binary decision with business costs.
3. Choose a decision threshold from **asymmetric costs** using out-of-fold predictions and audit the savings on a locked test set.
4. Audit a model for **group fairness** (demographic parity, equal opportunity) with bootstrap uncertainty, and test for **proxy** variables.
5. Produce **reason codes** that explain an individual decision, and package a decision function with a model card.

> 💡 **How to use this notebook:** run cells top to bottom. Read the text *before* you run the code. When you reach a 🧩 **Problem**, try it yourself first, open the hints one at a time, and only then peek at the solutions notebook.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))      # so we can import course_utils.py
from course_utils import *                      # np, pd, plt, check(), data_path(), ...
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
RANDOM_STATE = 42
print("Setup complete ✅")
import warnings, joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, cross_val_predict, StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, f1_score, log_loss, confusion_matrix
warnings.filterwarnings("ignore")

Setup complete ✅


## 📦 The data: `data/loan_approval.csv` (3 000 synthetic applications)

| Column | Meaning |
|---|---|
| `loan_id` | identifier |
| `age`, `gender`, `married`, `dependents`, `education` | applicant profile (**`gender` and `married` are protected attributes**) |
| `employment_type`, `monthly_income_kes`, `property_area` | employment/location |
| `loan_amount_kes`, `loan_term_months` | the requested loan |
| `credit_score` (**some missing**), `credit_history_years`, `existing_loans`, `monthly_debt_kes` | credit profile |
| `has_collateral`, `savings_balance_kes` (**some missing**) | security & savings |
| `interest_rate_offered` | ⚠️ **look at this one carefully** |
| **`approved`** | **TARGET**: 1 = approved, 0 = declined (≈ 60 % approved) |

## 🎯 Deliverables
1. This notebook, fully run, with your reasoning in added markdown cells.
2. `loan_model.joblib` (pipeline + threshold) and a working `decide()` function returning a decision with probability and reasons.
3. A model card.

## 🧮 Rubric (100 points)
| Area | Pts | Excellent looks like |
|---|---|---|
| Leakage detection | 15 | Finds the leaky column, explains the mechanism, quantifies the fake gain |
| Pipeline & fair tournament | 20 | No leakage, shared folds, baseline, ≥ 5 models, AUC + F1 + log loss |
| Tuning | 5 | Small, sensible grid |
| Cost-based threshold | 15 | Out-of-fold choice, test-set savings reported |
| Fairness audit | 20 | Parity + equal opportunity with uncertainty; proxy check; reasoned decision on protected attributes |
| Reason codes | 10 | Correct, readable top-3 reasons for a decline |
| Packaging & model card | 15 | Reloadable, honest limitations |

## 🗺️ Milestones
1. Profile · 2. **Leakage hunt** · 3. Features & split · 4. Tournament · 5. Tune · 6. Cost-based threshold · 7. Fairness audit (a, b) · 8. Reason codes · 9. Package & model card

---
## 📥 Milestone 0: Shared helper code (given)

In [2]:
def load_loans():
    d = pd.read_csv(data_path("loan_approval.csv"))
    d["emi"] = d["loan_amount_kes"] / d["loan_term_months"] * 1.15                                   # rough monthly instalment
    d["dti"] = (d["monthly_debt_kes"] + d["emi"]) / d["monthly_income_kes"]                          # debt-to-income
    d["lti"] = d["loan_amount_kes"] / (d["monthly_income_kes"] * 12)                                 # loan-to-annual-income
    d["log_income"] = np.log(d["monthly_income_kes"]); d["log_savings"] = np.log1p(d["savings_balance_kes"])
    return d

NUM = ["credit_score", "credit_history_years", "dti", "lti", "log_income", "log_savings", "has_collateral", "age", "dependents", "existing_loans"]
CAT = ["employment_type", "property_area", "education"]
def make_prep(num=NUM, cat=CAT):
    return ColumnTransformer([("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), list(num)),
                              ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), list(cat))])
loans = load_loans(); print(loans.shape); display(loans.head(3))

(3000, 24)


,loan_id,age,gender,married,dependents,education,employment_type,monthly_income_kes,loan_amount_kes,loan_term_months,credit_score,credit_history_years,existing_loans,monthly_debt_kes,property_area,has_collateral,savings_balance_kes,interest_rate_offered,approved,emi,dti,lti,log_income,log_savings
0,LN100000,22,Female,No,2,Graduate,Salaried,185200.0,4.149e+06,24,649.0,7.7,1,7400.0,Urban,0,146000.0,NaN,0,198806.250,1.113,1.867,12.129,11.891
1,LN100001,62,Female,No,0,Graduate,Salaried,40600.0,1.040e+05,24,696.0,8.9,0,0.0,Rural,0,176600.0,17.94,1,4983.333,0.123,0.213,10.612,12.082
2,LN100002,62,Female,Yes,2,Not Graduate,Business Owner,55500.0,1.200e+06,120,620.0,21.1,1,3200.0,Urban,1,NaN,17.82,1,11500.000,0.265,1.802,10.924,NaN


---
## 🔍 Milestone 1: Profile the data (part of 'Pipeline & tournament')

---
### 🧩 Problem 1: Profile  ⭐

`n_rows`, `approval_rate` (float), and `missing_cols`: the list of columns with missing values (note: the engineered `log_savings` inherits the gaps of `savings_balance_kes`, so it will appear too).

In [3]:
# ✅ SOLUTION
n_rows = len(loans); approval_rate = float(loans["approved"].mean()); missing_cols = [c for c in loans.columns if loans[c].isna().any()]
print(n_rows, round(approval_rate, 3), missing_cols); print(loans[missing_cols].isna().sum().to_dict())

3000 0.604 ['credit_score', 'savings_balance_kes', 'interest_rate_offered', 'log_savings']
{'credit_score': 96, 'savings_balance_kes': 153, 'interest_rate_offered': 1188, 'log_savings': 153}


**Why this works.** `interest_rate_offered` is missing for ~40 % of rows: almost exactly the share of declined loans. Missingness that lines up with the target is a giant red flag.

In [4]:
check("3000 applications", lambda: n_rows == 3000)
check("about 60% approved", lambda: abs(approval_rate - 0.604) < 0.005)
check("credit_score, savings_balance_kes and interest_rate_offered have gaps (plus the derived log_savings)", lambda: {"credit_score", "savings_balance_kes", "interest_rate_offered"} <= set(missing_cols) <= {"credit_score", "savings_balance_kes", "interest_rate_offered", "log_savings"})

✅  3000 applications
✅  about 60% approved
✅  credit_score, savings_balance_kes and interest_rate_offered have gaps (plus the derived log_savings)


---
## 🕵️ Milestone 2: The leakage hunt (15 pts)

A loan is declined, so no interest rate is ever offered. **The rate is only recorded after the approval decision.** If we use it to predict approval, we're feeding the model the answer. Prove it, quantify how misleading it is, and decide what to drop.

---
### 🧩 Problem 2: Interrogate the suspicious column  ⭐⭐

1. `rate_missing_rejected`, `rate_missing_approved`: the share of **declined** / **approved** applications whose `interest_rate_offered` is missing.
2. `leak_auc`: the ROC-AUC obtained by using the single indicator `loans['interest_rate_offered'].notna()` as a predictor of `approved` (a one-feature 'model').
3. `leaky_cv_auc`: the 5-fold stratified CV (`StratifiedKFold(5, shuffle=True, random_state=0)`) ROC-AUC of `DecisionTreeClassifier(max_depth=2, random_state=0)` trained on just `credit_score` (median-filled) and `rate_filled` (= the rate with missing → 0).
4. `honest_cv_auc`: the same tree trained **without** the rate (`credit_score` and `dti` only).
5. `drop_cols`: the list of columns that must **not** be used as features: the id, the leaky column and (by policy) the protected attributes.

In [5]:
# ✅ SOLUTION
rate_missing_rejected = float(loans.loc[loans["approved"] == 0, "interest_rate_offered"].isna().mean()); rate_missing_approved = float(loans.loc[loans["approved"] == 1, "interest_rate_offered"].isna().mean())
leak_auc = roc_auc_score(loans["approved"], loans["interest_rate_offered"].notna())
skf = StratifiedKFold(5, shuffle=True, random_state=0); cs = loans["credit_score"].fillna(loans["credit_score"].median())
X_leaky = pd.DataFrame({"credit_score": cs, "rate_filled": loans["interest_rate_offered"].fillna(0)}); X_honest = pd.DataFrame({"credit_score": cs, "dti": loans["dti"]})
tree = DecisionTreeClassifier(max_depth=2, random_state=0)
leaky_cv_auc = cross_val_score(tree, X_leaky, loans["approved"], cv=skf, scoring="roc_auc").mean(); honest_cv_auc = cross_val_score(tree, X_honest, loans["approved"], cv=skf, scoring="roc_auc").mean()
drop_cols = ["loan_id", "interest_rate_offered", "gender", "married"]
print(rate_missing_rejected, rate_missing_approved, leak_auc, round(leaky_cv_auc, 3), round(honest_cv_auc, 3))

1.0 0.0 1.0 1.0 0.827


**Why this works.** A depth-2 tree 'predicts' approval perfectly from a column that only exists *after* approval: a fantasy AUC of 1.0 versus a realistic ≈ 0.83. In production the rate wouldn't exist for a new applicant, so the leaky model would collapse. Always ask of every feature: *would I have this value at decision time?*

In [6]:
check("every declined loan has no rate; every approved loan has one", lambda: rate_missing_rejected == 1.0 and rate_missing_approved == 0.0)
check("the missingness indicator alone is a perfect predictor (AUC = 1)", lambda: abs(leak_auc - 1.0) < 1e-9)
check("the leaky tree scores a perfect 1.0 CV AUC", lambda: leaky_cv_auc > 0.999)
check("the honest tree scores far lower (< 0.9)", lambda: honest_cv_auc < 0.9)
check("drop list contains id, leaky column and protected attributes", lambda: set(drop_cols) >= {"loan_id", "interest_rate_offered", "gender", "married"})

✅  every declined loan has no rate; every approved loan has one
✅  the missingness indicator alone is a perfect predictor (AUC = 1)
✅  the leaky tree scores a perfect 1.0 CV AUC
✅  the honest tree scores far lower (< 0.9)
✅  drop list contains id, leaky column and protected attributes


✍️ *In a markdown cell, explain in your own words (3 sentences) what 'target leakage' is, how you spotted it here, and how you'd check for it on a new dataset.*

---
## 🛠️ Milestone 3: Features and the split

In [7]:
# 🔄 Checkpoint
loans = load_loans()

---
### 🧩 Problem 3: Build X, y and the split  ⭐⭐

1. `X = loans[NUM + CAT]` and `y = loans['approved']` (protected attributes and the leaky/id columns stay **out**); keep `prot = loans[['gender', 'married']]` separately for the fairness audit.
2. A **stratified** 75/25 split with `random_state=42` → `Xtr, Xte, ytr, yte`.
3. `prep = make_prep()` fitted inside pipelines later; verify `prep.fit_transform(Xtr)` has no NaN.

In [8]:
# ✅ SOLUTION
X = loans[NUM + CAT]; y = loans["approved"]; prot = loans[["gender", "married"]]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
prep = make_prep(); print(Xtr.shape, Xte.shape, round(ytr.mean(), 3), round(yte.mean(), 3))

(2250, 13) (750, 13) 0.604 0.604


**Why this works.** `stratify=y` keeps the approval rate identical in train and test, which matters for stable metrics. The protected attributes live in a separate frame so we can audit fairness *without* letting the model use them.

In [9]:
check("no leaky / id / protected columns in X", lambda: not ({"loan_id", "interest_rate_offered", "gender", "married"} & set(X.columns)))
check("engineered ratios dti and lti are present", lambda: {"dti", "lti"} <= set(X.columns))
check("stratified split: 2250 / 750 with equal approval rates", lambda: len(Xtr) == 2250 and len(Xte) == 750 and abs(ytr.mean() - yte.mean()) < 0.005)
check("preprocessing output has no NaN", lambda: not np.isnan(make_prep().fit_transform(Xtr)).any())

✅  no leaky / id / protected columns in X
✅  engineered ratios dti and lti are present
✅  stratified split: 2250 / 750 with equal approval rates
✅  preprocessing output has no NaN


---
## 🥊 Milestone 4: The tournament (20 pts together with Milestone 1)

A baseline plus **at least five** classifiers from the toolbox (Logistic Regression, Decision Tree, Random Forest, KNN, Naive Bayes, SVM…) in `Pipeline([('prep', prep), ('model', est)])`, with identical stratified CV (`StratifiedKFold(5, shuffle=True, random_state=0)`) and **three metrics**: ROC-AUC, F1 and log loss.

In [10]:
# 🔄 Checkpoint
loans = load_loans(); X = loans[NUM + CAT]; y = loans["approved"]; prot = loans[["gender", "married"]]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
prep = make_prep(); skf = StratifiedKFold(5, shuffle=True, random_state=0)

---
### 🧩 Problem 4: Run the tournament  ⭐⭐⭐

Build `models` (dict) and `leader`: a DataFrame indexed by model name with columns `cv_auc`, `cv_f1`, `cv_logloss` (positive), sorted by `cv_auc` descending, and `champion` (name of the best by AUC). Include `DummyClassifier(strategy='prior')` as the baseline.

In [11]:
# ✅ SOLUTION
mk = lambda est: Pipeline([("prep", prep), ("model", est)])
models = {"Baseline (prior)": mk(DummyClassifier(strategy="prior")), "Logistic Regression": mk(LogisticRegression(max_iter=2000)), "Decision Tree": mk(DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=0)),
          "Random Forest": mk(RandomForestClassifier(300, min_samples_leaf=5, random_state=0, n_jobs=-1)), "KNN (k=15)": mk(KNeighborsClassifier(15)), "Naive Bayes": mk(GaussianNB()), "SVM (RBF)": mk(SVC(probability=True, random_state=0))}
rows = {}
for n, m in models.items():
    r = cross_validate(m, Xtr, ytr, cv=skf, scoring={"auc": "roc_auc", "f1": "f1", "nll": "neg_log_loss"})
    rows[n] = {"cv_auc": r["test_auc"].mean(), "cv_f1": r["test_f1"].mean(), "cv_logloss": -r["test_nll"].mean()}
leader = pd.DataFrame(rows).T.sort_values("cv_auc", ascending=False); champion = leader.index[0]; display(leader.round(3)); print("champion:", champion)

,cv_auc,cv_f1,cv_logloss
Logistic Regression,0.930,0.887,0.327
SVM (RBF),0.918,0.880,0.351
Random Forest,0.916,0.885,0.381
Naive Bayes,0.905,0.868,0.405
KNN (k=15),0.881,0.850,0.499
Decision Tree,0.880,0.854,0.652
Baseline (prior),0.500,0.753,0.671


champion: Logistic Regression


**Why this works.** Logistic regression leads (AUC ≈ 0.93) with the SVM (0.92) and the forest (0.92) close behind; the single tree and KNN trail, and the tree's log loss is no better than the do-nothing baseline's. With engineered ratios the problem is nearly linear in the log-odds, so the simple, explainable model is also the best: good news for a regulated lender.

In [12]:
check("baseline plus >= 5 real models", lambda: len(models) >= 6)
check("leader has the three metrics, sorted by AUC", lambda: {"cv_auc", "cv_f1", "cv_logloss"} <= set(leader.columns) and leader["cv_auc"].is_monotonic_decreasing)
check("baseline AUC = 0.5", lambda: abs(leader[[("aseline" in i) for i in leader.index]]["cv_auc"].iloc[0] - 0.5) < 1e-9)
check("champion AUC > 0.92", lambda: leader["cv_auc"].iloc[0] > 0.92)
check("log loss is positive", lambda: (leader["cv_logloss"] > 0).all())

✅  baseline plus >= 5 real models
✅  leader has the three metrics, sorted by AUC
✅  baseline AUC = 0.5
✅  champion AUC > 0.92
✅  log loss is positive


---
## 🎛️ Milestone 5: Tune (5 pts)

In [13]:
# 🔄 Checkpoint
loans = load_loans(); X = loans[NUM + CAT]; y = loans["approved"]; prot = loans[["gender", "married"]]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
prep = make_prep(); skf = StratifiedKFold(5, shuffle=True, random_state=0)

---
### 🧩 Problem 5: Grid-search your champion  ⭐

Tune your champion family with `GridSearchCV` (3-fold stratified, `scoring='roc_auc'`). For logistic regression use `model__C ∈ [0.01, 0.1, 1, 10]` (and optionally the penalty). Store `tuned_model` (fitted best estimator on `Xtr, ytr`), `grid` (your parameter grid) and `tuned_cv_auc` = its 5-fold CV AUC with the shared `skf`.

In [14]:
# ✅ SOLUTION
grid = {"model__C": [0.01, 0.1, 1, 10]}
gs = GridSearchCV(Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000))]), grid, cv=StratifiedKFold(3, shuffle=True, random_state=1), scoring="roc_auc").fit(Xtr, ytr)
tuned_model = gs.best_estimator_; tuned_cv_auc = cross_val_score(tuned_model, Xtr, ytr, cv=skf, scoring="roc_auc").mean(); print(gs.best_params_, round(tuned_cv_auc, 4))

{'model__C': 1} 0.9298


**Why this works.** The grid barely matters for logistic regression (all C ≥ 0.1 are equivalent here), but running it documents that you checked.

In [15]:
check("a grid with at least 3 values was searched", lambda: sum(len(v) for v in grid.values()) >= 3)
check("tuned model is fitted", lambda: len(tuned_model.predict_proba(Xte)) == len(Xte))
check("tuned CV AUC > 0.925", lambda: tuned_cv_auc > 0.925)

✅  a grid with at least 3 values was searched
✅  tuned model is fitted
✅  tuned CV AUC > 0.925


---
## 💰 Milestone 6: Costs → threshold (15 pts)

Ms. Wambui's cost matrix: approving a loan that should have been declined (**FP**) costs **5**; declining a good applicant (**FN**) costs **1**. Choose τ from **out-of-fold** training predictions (never the test set), then report the **test** cost at the default τ = 0.5 versus your τ, and the resulting confusion matrix.

In [16]:
# 🔄 Checkpoint: a reference tuned model
loans = load_loans(); X = loans[NUM + CAT]; y = loans["approved"]; prot = loans[["gender", "married"]]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
prep = make_prep(); skf = StratifiedKFold(5, shuffle=True, random_state=0); taus = np.linspace(0.05, 0.95, 91)
ref_model = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000, C=1.0))])
def total_cost(y_true, p, tau, c_fp=5, c_fn=1):
    pred = p >= tau; return c_fp * np.sum(pred & (y_true == 0)) + c_fn * np.sum(~pred & (y_true == 1))

---
### 🧩 Problem 6: Optimal threshold and test savings  ⭐⭐⭐

Using `ref_model` (or your `tuned_model`):
1. `oof` = out-of-fold P(approve) on the training set (`cross_val_predict(..., cv=skf, method='predict_proba')[:, 1]`).
2. `tau_star` = the threshold in `taus` that minimises `total_cost(ytr.values, oof, tau)`.
3. Fit the model on all training data; on the **test set**: `cost_default` (τ = 0.5), `cost_tuned` (τ = `tau_star`), `saving_pct`, and `approve_rate_tuned` (share of test applicants approved at `tau_star`).
4. `cm_tuned`: the confusion matrix `[[TN, FP], [FN, TP]]` at `tau_star`.

In [17]:
# ✅ SOLUTION
final = ref_model; oof = cross_val_predict(final, Xtr, ytr, cv=skf, method="predict_proba")[:, 1]
tau_star = float(taus[int(np.argmin([total_cost(ytr.values, oof, t) for t in taus]))])
final.fit(Xtr, ytr); p_test = final.predict_proba(Xte)[:, 1]
cost_default = int(total_cost(yte.values, p_test, 0.5)); cost_tuned = int(total_cost(yte.values, p_test, tau_star)); saving_pct = 100 * (cost_default - cost_tuned) / cost_default
approve_rate_tuned = float(np.mean(p_test >= tau_star)); cm_tuned = confusion_matrix(yte, p_test >= tau_star)
print(round(tau_star, 2), cost_default, cost_tuned, round(saving_pct, 1), round(approve_rate_tuned, 3)); print(cm_tuned)

0.77 386 278 28.0 0.505
[[263  34]
 [108 345]]


**Why this works.** At τ* ≈ 0.77 the bank approves roughly half the applicants (vs 60 % historically): fewer costly bad approvals (FP ↓) at the price of more good applicants declined (FN ↑). Net cost falls by ≈ 28 %. Ms. Wambui must understand that these savings are conditional on her 5:1 cost assumption.

In [18]:
check("tau* is strict (between 0.6 and 0.9) because false approvals cost 5x", lambda: 0.6 < tau_star < 0.9)
check("the tuned threshold cuts test cost by more than 15%", lambda: saving_pct > 15)
check("the policy approves fewer applicants than the 60% historical rate", lambda: approve_rate_tuned < 0.6)
check("confusion matrix is 2x2 and sums to the test size", lambda: cm_tuned.shape == (2, 2) and cm_tuned.sum() == len(yte))

✅  tau* is strict (between 0.6 and 0.9) because false approvals cost 5x
✅  the tuned threshold cuts test cost by more than 15%
✅  the policy approves fewer applicants than the 60% historical rate
✅  confusion matrix is 2x2 and sums to the test size


---
## ⚖️ Milestone 7: Fairness audit (20 pts)

Regulators will ask two questions. **Demographic parity:** are approval *rates* similar across groups? **Equal opportunity:** among applicants who *deserve* approval (true approvals), is the chance of being approved similar (equal true-positive rates)? Audit **gender** at your threshold with **uncertainty** (bootstrap), then test whether other features act as **proxies** for the protected attribute.

In [19]:
# 🔄 Checkpoint: reference predictions and threshold
loans = load_loans(); X = loans[NUM + CAT]; y = loans["approved"]; prot = loans[["gender", "married"]]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
prep = make_prep(); skf = StratifiedKFold(5, shuffle=True, random_state=0)
clf = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000))]).fit(Xtr, ytr)
p_test = clf.predict_proba(Xte)[:, 1]; tau_star = 0.77; approved_pred = p_test >= tau_star; prot_te = prot.loc[Xte.index]

---
### 🧩 Problem 7: Group metrics with uncertainty  ⭐⭐⭐

With `approved_pred`, `yte` and `prot_te['gender']` (values 'Female'/'Male'):
1. `approve_rate`: dict gender → share predicted-approved; `dp_gap` = |difference| (demographic parity gap).
2. `tpr`: dict gender → share predicted-approved among truly approved (`yte == 1`); `eo_gap` = |difference|.
3. `tpr_ci`: bootstrap 95 % CI (2.5th, 97.5th percentile) of TPR(Female) − TPR(Male): 2000 resamples **with replacement within each group's truly-approved test applicants**, `rng = np.random.default_rng(0)`.
4. `eo_significant`: True if the CI excludes 0.

In [20]:
# ✅ SOLUTION
g = prot_te["gender"].values; yv = yte.values
approve_rate = {k: float(approved_pred[g == k].mean()) for k in ["Female", "Male"]}; dp_gap = abs(approve_rate["Female"] - approve_rate["Male"])
tpr = {k: float(approved_pred[(g == k) & (yv == 1)].mean()) for k in ["Female", "Male"]}; eo_gap = abs(tpr["Female"] - tpr["Male"])
rng = np.random.default_rng(0); fF, fM = approved_pred[(g == "Female") & (yv == 1)], approved_pred[(g == "Male") & (yv == 1)]
diffs = [rng.choice(fF, len(fF)).mean() - rng.choice(fM, len(fM)).mean() for _ in range(2000)]
tpr_ci = tuple(float(v) for v in np.percentile(diffs, [2.5, 97.5])); eo_significant = bool(tpr_ci[0] > 0 or tpr_ci[1] < 0)
print(approve_rate, round(dp_gap, 3), tpr, round(eo_gap, 3), [round(v, 3) for v in tpr_ci], eo_significant)

{'Female': 0.5163934426229508, 'Male': 0.4947916666666667} 0.022 {'Female': 0.7818181818181819, 'Male': 0.7424892703862661} 0.039 [-0.034, 0.117] False


**Why this works.** Approval rates differ by ≈ 2 points and TPRs by ≈ 4 points; the bootstrap interval for the TPR gap (about −3 to +12 points) straddles zero. With ~225 truly-approved applicants per gender the audit has decent power, so we can report: no evidence of a gender gap: consistent with the data, in which gender plays no role.

In [21]:
check("approval rates computed for both genders", lambda: set(approve_rate) == {"Female", "Male"})
check("demographic parity gap is small (< 0.06)", lambda: dp_gap < 0.06)
check("TPRs are similar (equal-opportunity gap < 0.1)", lambda: eo_gap < 0.1)
check("the bootstrap CI contains 0: no significant gap", lambda: tpr_ci[0] < 0 < tpr_ci[1] and eo_significant is False)

✅  approval rates computed for both genders
✅  demographic parity gap is small (< 0.06)
✅  TPRs are similar (equal-opportunity gap < 0.1)
✅  the bootstrap CI contains 0: no significant gap


---
### 🧩 Problem 8: Proxy test and the decision on protected attributes  ⭐⭐⭐

1. **Proxy test:** can the protected attribute be *predicted* from the allowed features? `proxy_auc` = CV ROC-AUC (`skf`) of a logistic-regression pipeline (`make_prep()`) predicting `is_female = (loans['gender'] == 'Female')` from `Xtr` (use `is_female.loc[Xtr.index]`). A value near 0.5 means no proxy.
2. **Does including the attribute help?** `auc_without` = CV AUC of the approval model on `Xtr` (as in Milestone 4); `auc_with` = CV AUC when `gender` and `married` (dummy-encoded, from `loans`) are added as extra categorical columns. `auc_gain = auc_with − auc_without`.
3. `use_protected` = `True`/`False`: your *recommendation* (hint: legal risk without predictive benefit).

In [22]:
# ✅ SOLUTION
is_female = (loans["gender"] == "Female").astype(int)
lr_ = lambda p: Pipeline([("prep", p), ("model", LogisticRegression(max_iter=2000))])
proxy_auc = cross_val_score(lr_(make_prep()), Xtr, is_female.loc[Xtr.index], cv=skf, scoring="roc_auc").mean()
auc_without = cross_val_score(lr_(make_prep()), Xtr, ytr, cv=skf, scoring="roc_auc").mean()
Xp = loans.loc[Xtr.index, NUM + CAT + ["gender", "married"]]
auc_with = cross_val_score(lr_(make_prep(cat=CAT + ["gender", "married"])), Xp, ytr, cv=skf, scoring="roc_auc").mean()
auc_gain = auc_with - auc_without; use_protected = False
print(round(proxy_auc, 3), round(auc_without, 4), round(auc_with, 4), round(auc_gain, 5), use_protected)

0.516 0.9298 0.9295 -0.00027 False


**Why this works.** Nothing in the allowed features predicts gender (AUC ≈ 0.52) and adding gender/marital status doesn't improve AUC. Using them would add legal and ethical risk for zero benefit: the decision is clear. (In real data proxies such as postcode or occupation often *do* leak protected information; this test is how you find them.)

In [23]:
check("no proxy: gender cannot be predicted from the allowed features (AUC < 0.56)", lambda: proxy_auc < 0.56)
check("including protected attributes adds nothing (|gain| < 0.003)", lambda: abs(auc_gain) < 0.003)
check("recommendation: do not use protected attributes", lambda: use_protected is False)

✅  no proxy: gender cannot be predicted from the allowed features (AUC < 0.56)
✅  including protected attributes adds nothing (|gain| < 0.003)
✅  recommendation: do not use protected attributes


✍️ *Write a short paragraph for the regulator: what you measured, what you found, and the limits of your audit (only two attributes, one threshold, synthetic data, historical labels that may themselves be biased).*

---
## 🗣️ Milestone 8: Reason codes (10 pts)

Regulators and customers deserve an explanation for each decline. For a **logistic regression** on standardised features, the score is a sum of contributions `coef × transformed_value`; the most **negative** contributions are the reasons pushing towards decline.

In [24]:
# 🔄 Checkpoint: fitted reference model and a declined applicant
loans = load_loans(); X = loans[NUM + CAT]; y = loans["approved"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
clf = Pipeline([("prep", make_prep()), ("model", LogisticRegression(max_iter=2000))]).fit(Xtr, ytr)
p_test = clf.predict_proba(Xte)[:, 1]; tau_star = 0.77; declined_idx = int(np.where(p_test < tau_star)[0][0])
print("first declined test applicant (position):", declined_idx, "| P(approve) =", round(p_test[declined_idx], 3))

first declined test applicant (position): 0 | P(approve) = 0.215


---
### 🧩 Problem 9: Top-3 reasons for a decline  ⭐⭐

Write `reason_codes(pipeline, x_row, k=3)` returning the names (strip the `num__`/`cat__` prefix) of the `k` features with the **most negative** contribution `coef × transformed value` for the one-row DataFrame `x_row`. Apply it to the declined applicant `Xte.iloc[[declined_idx]]` → `reasons`. Set `reasons_ok` = `True` if at least two of the reasons are among `{'dti', 'lti', 'credit_score', 'has_collateral'}`.

In [25]:
# ✅ SOLUTION
def reason_codes(pipeline, x_row, k=3):
    names = pipeline[0].get_feature_names_out(); contrib = pipeline[-1].coef_[0] * pipeline[0].transform(x_row)[0]
    return [names[i].split("__", 1)[1] for i in np.argsort(contrib)[:k]]

reasons = reason_codes(clf, Xte.iloc[[declined_idx]]); reasons_ok = len({"dti", "lti", "credit_score", "has_collateral"} & set(reasons)) >= 2
print(reasons, reasons_ok)

['lti', 'has_collateral', 'credit_score'] True


**Why this works.** For this applicant the biggest negatives are a high loan-to-income ratio, no collateral and a below-average credit score: exactly the kind of statement a decline letter needs ('your requested amount is large relative to your income…').

In [26]:
check("three reasons returned", lambda: isinstance(reasons, list) and len(reasons) == 3)
check("names are clean (no prefixes)", lambda: all("__" not in r for r in reasons))
check("at least two reasons are core credit-risk drivers", lambda: reasons_ok)

✅  three reasons returned
✅  names are clean (no prefixes)
✅  at least two reasons are core credit-risk drivers


---
## 📦 Milestone 9: Package and document (15 pts)

Deliver something the loan officers can call: a saved model **plus** threshold, and a `decide()` function returning the decision, the probability and the reasons.

In [27]:
# 🔄 Checkpoint: reference model/threshold/reason function
loans = load_loans(); X = loans[NUM + CAT]; y = loans["approved"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
clf = Pipeline([("prep", make_prep()), ("model", LogisticRegression(max_iter=2000))]).fit(Xtr, ytr); tau_star = 0.77
def reason_codes(pipeline, x_row, k=3):
    names = pipeline[0].get_feature_names_out(); contrib = pipeline[-1].coef_[0] * pipeline[0].transform(x_row)[0]
    return [names[i].split("__", 1)[1] for i in np.argsort(contrib)[:k]]

---
### 🧩 Problem 10: decide() and the model card  ⭐⭐⭐

1. Save `{'model': clf, 'threshold': tau_star}` with `joblib.dump` to `'loan_model.joblib'`; reload as `bundle`.
2. Write `decide(applicant, bundle)` for a one-row DataFrame (columns of `X`) returning a dict `{'decision': 'APPROVE' or 'DECLINE', 'p_approve': float, 'reasons': list}`: reasons only for declines (else `[]`).
3. Test it on two applicants: `d_best = decide(...)` for the **most creditworthy** test applicant (highest P(approve)) and `d_declined = decide(...)` for the first declined one (the first test row with `p < tau`, as in Milestone 8).
4. Fill `model_card` (keys `intended_use`, `data`, `metrics`, `fairness`, `limitations`) with honest sentences.

In [28]:
# ✅ SOLUTION
joblib.dump({"model": clf, "threshold": tau_star}, "loan_model.joblib"); bundle = joblib.load("loan_model.joblib")
def decide(applicant, bundle):
    p = float(bundle["model"].predict_proba(applicant)[0, 1]); approve = p >= bundle["threshold"]
    return {"decision": "APPROVE" if approve else "DECLINE", "p_approve": p, "reasons": [] if approve else reason_codes(bundle["model"], applicant)}

p_all = clf.predict_proba(Xte)[:, 1]; declined_idx = int(np.where(p_all < tau_star)[0][0])
d_best = decide(Xte.iloc[[int(np.argmax(p_all))]], bundle); d_declined = decide(Xte.iloc[[declined_idx]], bundle)
model_card = {"intended_use": "Pre-screening aid that recommends approve/decline so officers can focus on borderline cases; not an automatic final decision.",
              "data": "3 000 synthetic applications; leaky column interest_rate_offered, the id and protected attributes were excluded; features include engineered debt-to-income and loan-to-income ratios.",
              "metrics": "Logistic regression: CV ROC-AUC ≈ 0.93; at the cost-optimal threshold ≈ 0.77 (false approval costs 5x a false decline) the test cost fell by ≈ 28% versus a 0.5 threshold.",
              "fairness": "Gender: approval-rate gap ≈ 2 points, equal-opportunity gap ≈ 4 points with a bootstrap CI that includes zero; no feature acts as a gender proxy; protected attributes are not used.",
              "limitations": "Synthetic data; historical approvals may encode past bias; audit covered only gender and marital status at one threshold; cost ratio of 5:1 is an assumption to be revisited; the model must be monitored for drift."}
print(d_best); print(d_declined)

{'decision': 'APPROVE', 'p_approve': 0.9995292011600615, 'reasons': []}
{'decision': 'DECLINE', 'p_approve': 0.2146846853304386, 'reasons': ['lti', 'has_collateral', 'credit_score']}


**Why this works.** The bundle (pipeline + threshold) travels together, so the cost-based decision rule is never separated from the model it was tuned for. `decide()` returns what a loan officer needs: a recommendation, a probability and (for declines) human-readable reasons.

In [29]:
import os
check("bundle saved and reloaded", lambda: os.path.exists("loan_model.joblib") and set(bundle) == {"model", "threshold"})
check("decide returns the three fields, and the best applicant is approved", lambda: set(d_best) == {"decision", "p_approve", "reasons"} and d_best["decision"] == "APPROVE")
check("a declined applicant gets exactly three reasons", lambda: d_declined["decision"] == "DECLINE" and len(d_declined["reasons"]) == 3)
check("approved applicants get no reasons", lambda: d_best["reasons"] == [])
check("model card has five honest sections", lambda: set(model_card) == {"intended_use", "data", "metrics", "fairness", "limitations"} and all(isinstance(v, str) and len(v) > 40 for v in model_card.values()))

✅  bundle saved and reloaded
✅  decide returns the three fields, and the best applicant is approved
✅  a declined applicant gets exactly three reasons
✅  approved applicants get no reasons
✅  model card has five honest sections


## 🚀 Stretch goals
- **Calibration:** plot a reliability curve and compare logistic regression, a forest and a calibrated forest (`CalibratedClassifierCV`). Does τ* move toward theory (5/6)?
- **Reject inference:** the historical approvals are *decisions*, not repayment outcomes. Discuss how this biases the data and what extra data (actual defaults) would let you model risk directly.
- **Fairness mitigation:** compute group-specific thresholds that equalise TPR, and discuss whether that is legal/desirable; compare costs.
- **Monitoring plan:** which three quantities would you track monthly to detect drift (score distribution, approval rate by group, realised default rate)?
- **Counterfactual advice:** for a declined applicant, find the smallest change in `loan_amount_kes` that would flip the decision (a simple 'what would it take?' search).

## ✅ Self-assessment
- [ ] Did I catch and *explain* the leaky column?  - [ ] Is every threshold decision based on out-of-fold training data?
- [ ] Did I report fairness with uncertainty, and say what the audit cannot show?  - [ ] Could a declined customer understand my reason codes?

---
## 🔁 Recap

- Leakage (a column recorded after the decision) can give a perfect-looking, useless model: interrogate every suspicious column.
- Asymmetric costs (5:1) move the optimal threshold to ≈ 0.77 and cut cost by ≈ 28 %; always choose it on out-of-fold data.
- Fairness needs metrics *and* uncertainty *and* a proxy check; protected attributes added nothing, so excluding them is the defensible policy.
- Reason codes and a model card make a model usable, and accountable.

### ➡️ Up next: **Project 4: Agricultural Price Regression**
Open `P4_Agricultural_Price_Regression.ipynb` when you're ready.

*You did the hard part: you thought about it, not just ran it. See you in the next one!* 🚀